# Results Interpretation

This notebook synthesizes the project in an academic forecasting-paper style.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import CLEAN_SEGMENTS, FIGURES, TABLES, TARGET_COLUMNS, SEGMENT_COLUMNS, SERIES_COLORS, SERIES_LABELS
from src.plotting import annotate_events, save_figure, set_academic_style

set_academic_style()
df = pd.read_csv(CLEAN_SEGMENTS, parse_dates=["date"]).set_index("date").sort_index()
df.index.freq = "MS"

from src.models.evaluation import run_all
validation = pd.read_csv(TABLES / 'data_validation_checks.csv')
metrics = pd.read_csv(TABLES / 'model_metrics.csv') if (TABLES / 'model_metrics.csv').exists() else run_all(df)[0]
best = pd.read_csv(TABLES / 'best_models.csv') if (TABLES / 'best_models.csv').exists() else run_all(df)[1]

## Research Objective

The objective is to forecast Vietnam's monthly international tourist arrivals using VNAT segment data, with attention to seasonality, volatility, and structural breaks.

In [ ]:
validation

## Data Source and Variables

The dataset is `data/raw/vnat_monthly_segments.csv`, cleaned to `data/processed/vnat_monthly_segments_clean.csv`. Variables include total arrivals and five regional segments: Asia, Europe, Americas, Oceania, and other markets. ASEAN is not used.

## EDA Findings

Observation: arrivals are seasonal, shock-sensitive, and compositionally uneven. Statistical implication: models need seasonal structure and robust post-shock evaluation. Tourism implication: planning should combine aggregate and regional evidence.

In [ ]:
fig, ax = plt.subplots()
ax.plot(df.index, df["international_arrivals"], color=SERIES_COLORS["international_arrivals"])
annotate_events(ax, ["COVID-19 outbreak", "Vietnam border reopening", "Russia-Ukraine war", "Iran-Israel conflict"])
ax.set_title("Vietnam Tourism Demand Under Global and Domestic Shocks")
ax.set_ylabel("Monthly arrivals")
save_figure(fig, FIGURES / "15_final_narrative_series.png")
plt.show()

## Model Performance

Observation: out-of-sample accuracy differs across model classes and segments. Statistical implication: no single specification should be assumed dominant without test-period evidence. Tourism implication: model choice should be revised when recovery dynamics change.

In [ ]:
metrics.round(2)

In [ ]:
best[['target', 'model', 'MAE', 'RMSE', 'MAPE', 'sMAPE']].round(2)

## Forecast Interpretation

Observation: the recovery path remains volatile despite the return of strong arrival volumes. Statistical implication: confidence intervals and residual diagnostics are central outputs. Tourism implication: policymakers and firms should treat forecasts as risk ranges for capacity, promotion, and infrastructure decisions.

## Limitations and Future Improvements

The analysis depends on crawler-derived VNAT data and deterministic imputation for missing or stale observations. Future work should refresh raw VNAT pages, incorporate exogenous indicators such as flight capacity and exchange rates, and test structural-break or regime-switching models.